# RAG Pipeline — Вопрос-ответ по PDF-документам

Ноутбук реализует RAG (Retrieval-Augmented Generation) — систему, которая ищет релевантные фрагменты в загруженных PDF-файлах и генерирует ответ с помощью GPT.

**Этап 1 — Индексация (выполняется один раз):**
1. **Загрузка** — читаем PDF-файлы
2. **Разбиение** — делим текст на фрагменты (chunks)
3. **Эмбеддинг** — переводим фрагменты в векторное представление
4. **Хранение** — сохраняем векторы в ChromaDB

**Этап 2 — Поиск и генерация ответа (при каждом вопросе):**
1. Принимаем вопрос от пользователя
2. Переводим вопрос в вектор
3. Находим Top-K (кол-во chunks сколько задали) наиболее похожих фрагментов в ChromaDB
4. Отправляем фрагменты + вопрос в GPT для генерации ответа











In [ ]:
# Установка библиотек для RAG-системы
# langchain_community — загрузчик PDF-файлов
# langchain==0.1.10   — интеграция с GPT и инструменты RAG
# chromadb            — локальная векторная БД
# pypdf               — чтение текста из PDF
# tiktoken            — токенизатор OpenAI (подсчёт токенов, разбиение текста)
!pip install langchain_community langchain==0.1.10 chromadb pypdf tiktoken

## Конфигурация

Создание файла config.json для сохранения ключей (gpt) - для вызова API

После выполнения ячейки файл появится в панели **Файлы** (иконка папки слева).

In [ ]:
# Библиотека для создания файла и записи текста в него
from pathlib import Path
# Библиотека для работы с json
import json
# Создание файла `config.json`

config_path = Path("config.json")
config_data = {
  "provider": "openai",
  "base_url": "https://api.openai.com/v1",
  "llm_model": "gpt-4o-mini",
  "timeout_sec": 60
}

config_path.write_text(json.dumps(config_data, ensure_ascii=False, indent=2), encoding="utf-8")

## API-ключ

Сохраните ключ OpenAI в Secrets Google Colab:
- **Название:** `openai_api_key`  
- **Значение:** ваш ключ (получить на https://platform.openai.com/)

Панель Secrets — иконка 🔑 слева.

In [ ]:
# Библиотека для получения ключа в Google Colab
from google.colab import userdata
# Получение ключа из Секрета
api_key = userdata.get('openai_api_key')

In [ ]:
# Библиотека для работы с переменными
import os
# Создаем переменную config с данными из config.json
config = json.loads(
    Path("config.json").read_text(encoding="utf-8")
)
# Создаем временные переменные (environ) для вызова GPT
os.environ['LLM_BASE_URL'] = config.get("base_url")
os.environ['LLM_MODEL'] = config.get("llm_model")
os.environ["LLM_API_KEY"] = api_key


In [ ]:
# Библиотеки для работы с OpenAI
from openai import OpenAI

In [ ]:
# Тестовый вызов GPT - https://developers.openai.com/api/docs/guides/text?lang=python
client = OpenAI(
    api_key=os.environ.get("LLM_API_KEY"),
    base_url=os.environ.get("LLM_BASE_URL")


)
response = client.responses.create(
    model=os.environ.get('LLM_MODEL'),
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

## Загрузка PDF-файлов

Загрузите zip-архив с PDF-файлами в панель **Файлы** и укажите его название в команде `unzip` (пример: Архив.zip).



In [ ]:
# Разархивирование архива
! unzip "/content/Архив.zip"
# Файлы появятся в "Файлы"


Создайте папку `Файлы_pdf` в панели **Файлы** и перенесите туда PDF-файлы.  
Рекомендуется добавить минимум 2 файла.

Создайте папку Файлы_pdf в панели Файлы и перенесите туда PDF-файлы.

In [ ]:
# Библиотека для поиска всех файлов в папке
from glob import glob
# Библиотека для загрузки PDF и чтение текста
from langchain_community.document_loaders import PyPDFLoader
# Путь к папке, где файлы
DOC_FOLDER = "/content/Файлы_pdf/"
# Ищем все файлы в папке с расширением pdf
pdf_files = glob(DOC_FOLDER + "*.pdf")
# Массив для сбора всех страниц документов pdf
all_pages = []
# цикл для сбора всех страниц всех найденных pdf
for pdf_path in pdf_files:
    # создаёт loader для PDF
    loader = PyPDFLoader(pdf_path)
    # извлекает текст страниц PDF
    pages = loader.load()
    # extend - добавляет страницы в общий список
    all_pages.extend(pages)

In [ ]:
# тест, что файлы объединились. просто посмотреть, что массив не пустой
print(all_pages)

## Разбиение на чанки (chunks)

Процесс поиска по большому файлу будет долгий, поэтому делим на фрагменты

chunk_size - максимальное число символов в одном фрагменте

chunk_overlap - сколько символов пересекается из предыдущего чанка в следующий, чтобы не потерять контекст. Рекомендуется 10–20% от `chunk_size`

In [ ]:
# Библиотека разделения большого текста на маленькие части
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Создаём объект text_splitter, с настройками разделения
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
# делим объеденный текст (all_pages) на чанки
chunks = text_splitter.split_documents(all_pages)

In [ ]:
# тест получения одного чанка
print(chunks[9])

## Создание векторной базы данных (ChromaDB)

Каждый чанк переводится в числовой вектор (эмбеддинг) с помощью модели OpenAI.  
Векторы сохраняются в ChromaDB — локальной базе, которая умеет искать похожие фрагменты по близости векторов.  
Загрузка идёт батчами.


In [ ]:
# создаем переменную для сохранения или создания БД
CHROMA_PATH = "/content/vector_DB"
# импорт класс OpenAIEmbeddings, нужен для преобразования чанков в вектора
from langchain.embeddings.openai import OpenAIEmbeddings
# Импортируем Chroma, для сохранения векторов Chroma DB
from langchain.vectorstores import Chroma
# Импортируем класс Document. для сохранения метаданных чанков (страницы, название файла и т.д.)
from langchain.docstore.document import Document
# Импортируем List, для удобства указания, что ожидаем список документов, а не один документ
from typing import List

# Создаём объект модели эмбеддингов OpenAI.
# Эта модель будет превращать текстовые фрагменты документации в векторы.
embeddings = OpenAIEmbeddings(
    openai_api_key=os.environ['LLM_API_KEY'],
    openai_api_base=os.environ['LLM_BASE_URL']
)


# загрузка в Chroma

# MAX_BATCH_SIZE - максимальный размер батча, кол-во чанков за раз, кол-во документов для загрузки за один раз.
# Если сделать слишком большой, будет ошибка ограничения размера batch
MAX_BATCH_SIZE = 16

# Функция для пакетной загрузки
def batch_insert_documents(
    # список чанков для сохранения
    documents: List[Document],
    # модель для эмбединга, для преобразования чанков (текста) в вектора
    embeddings: str,
    # путь для сохранения векторной БД
    persist_directory: str,
    # размер батча
    batch_size: int
) -> Chroma:
    # Подсчет общего кол-ва чанков для сохранения
    total_documents = len(documents)


    # объект векторной базы ChromaDB
    db_chroma = Chroma(
        # путь для сохранения векторной БД
        persist_directory=persist_directory,
        # модель для эмбединга
        embedding_function=embeddings,
        # коллекция в БД, куда сохраняем
        collection_name="temp_collection"
    )

    ''' Цикл загрузки в БД, от 0 до total_documents (считали кол-во выше) с batch_size
    кол-во документов за одну загрузку
    '''
    for i in range(0, total_documents, batch_size):

        # получаем количество чанков
        batch = documents[i:i + batch_size]

        # Преобразование текста в вектор и сохранение в ChromaDB
        db_chroma.add_documents(
            documents=batch,
        )
        print("текущий:", i, "всего:", total_documents)

# вызываем функцию
db = batch_insert_documents(
    documents=chunks,
    embeddings=embeddings,
    persist_directory=CHROMA_PATH,
    batch_size=MAX_BATCH_SIZE
)

## Поиск релевантных фрагментов

По вопросу пользователя ищем Top-K наиболее близких чанков в векторной БД.


In [ ]:
# Вопрос
user_input = "Расскажи о защите прав потребителей"
# Количество чанков в ответе поиска Top_k, k=
docs_chroma = db_chroma.similarity_search_with_score(user_input, k=5)
# Печать с метаданными
for i, (doc, _score) in enumerate(docs_chroma):
    print(f"Найденный фрагмент {i+1}:\n")
    print(doc)
    print(doc.page_content.replace('\t', ' '))
    print("Источник:", doc.metadata['source'], "\n")
    print("Номер страницы:", doc.metadata['page'])
    print("\n=====================================================\n")
    print("\n")

In [ ]:
# объединяем для отправки потом в гпт
context_text = "\n\n".join([doc.page_content for doc, _score in docs_chroma])

## Генерация ответа (GPT)

Передаём найденный контекст и вопрос пользователя в GPT.  
Модель отвечает строго на основе предоставленных фрагментов

In [ ]:
# для удобства подстановки значений {} в промт
from langchain.prompts import ChatPromptTemplate
# финальный ответ GPT на основе найденных chunks
from langchain.chat_models import ChatOpenAI

PROMPT_TEMPLATE = """Ты - AI-ассестент, для ответов на вопрос пользователя.
Твоя задача отвечать только операясь на контекст. Если ответа в контексте нет — скажи об этом.
Контекст: {context}
Вопрос пользователя: {question}"""

# Создаём объект шаблона промпта из строки PROMPT_TEMPLATE.
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

# Подставляем в шаблон:
prompt = prompt_template.format(
    context=context_text,
    question=user_input
)

# Итоговый промт, который отправится в гпт
print(prompt)

In [ ]:
# Вызов gpt
model = ChatOpenAI(
    model_name=config.get("llm_model"),
    api_key = api_key,
    base_url = config.get("base_url")
)
response_text = model.predict(prompt)
print(response_text)